In [0]:
from pyspark.sql.functions import col, trim, lower, when


# read bronze table
df = spark.table("bronze.data_dictionary")

# fix column names → lowercase
df = df.toDF(*[c.lower() for c in df.columns])

# cleaning (SAFE FORMAT WITH BRACKETS)
df = (
    df
    .withColumn("table", lower(trim(col("table"))))
    .withColumn("field", lower(trim(col("field"))))
    .withColumn("description", trim(col("description")))

    # handle blank + unknown → null
    .withColumn(
        "table",
        when((col("table") == "") | (col("table") == "unknown"), None)
        .otherwise(col("table"))
    )
    .withColumn(
        "field",
        when((col("field") == "") | (col("field") == "unknown"), None)
        .otherwise(col("field"))
    )
    .withColumn(
        "description",
        when((col("description") == "") | (col("description") == "unknown"), None)
        .otherwise(col("description"))
    )
)


# Remove system columns (like _line, _fivetran_synced)
df = df.select([
    col(c) for c in df.columns if not c.startswith("_")
])


# save to silver
df.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("silver.data_dictionary")